In [0]:
%sql
USE CATALOG ecommerce

## Products

In [0]:
%sql

CREATE OR REPLACE TABLE gold.gold_products AS
(
  SELECT
    p.product_id,
    p.sku,
    p.category_code,
    c.category_name,
    p.brand_code,
    b.brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.lenght_cm AS length_cm,
    p.widht_cm AS width_cm,
    p.height_cm,
    p.rating_count,
    p.`_souce_file`,
    p.ingested_at
  FROM silver.silver_products p
  LEFT JOIN silver.silver_brands b ON p.brand_code = b.brand_code
  LEFT JOIN silver.silver_category c ON p.category_code = c.category_code
)

## Customers

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
l = []
for country, state_map in country_state_map.items():
    for state, region in state_map.items():
        l.append([country, state, region])

df_regions = spark.createDataFrame(l, ["country", "state", "region"])
df_regions.show(5)

In [0]:
df_silver_customers = spark.table('ecommerce.silver.silver_customers')
df_gold_customers = df_silver_customers.join(df_regions, on=['country','state'], how='left')
df_gold_customers.write.mode('overwrite')\
    .format('delta')\
    .option('overwriteSchema', 'true')\
    .saveAsTable('ecommerce.gold.gold_customers')

## Date

In [0]:
%sql

CREATE OR REPLACE TABLE gold.gold_date AS
(
  SELECT
    TO_CHAR(date, 'yyyyMMdd') AS date_id,
    date,
    year,
    MONTHNAME(date) AS month_name,
    day_name,
    CASE 
      WHEN day_name IN ('Saturday', 'Sunday') THEN 1
      ELSE 0
    END AS is_weekend,
    quarter,
    week_of_year,
    `_souce_file`,
    ingested_at
  FROM silver.silver_date
)

In [0]:
%sql
SELECT * FROM gold.gold_date
LIMIT 5;